In [1]:
## Image processing for extracting BOM from engineering drawings. This programme is upto training of data. For taking out
## cropped image from actual drawings see another programme :
## Refer earlier programme 001_Kolkata_1_27_02_25.ipynb
# start date 27/02/2025

import numpy as np
import math
import pandas as pd
import cv2
import os
import re
from pdf2image import convert_from_path 
#import tqdm
#from scipy.io import loadmat

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

from PIL import Image
# import pytesseract

import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from keras import backend as K

# from utils import *

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from keras.layers import *

# from keras.applications import MobileNetV2
# from keras.applications import InceptionResNetV2

from keras.models import Model
from keras.models import model_from_json
from keras import regularizers

#from keras.initializers import he_normal

from keras.models import load_model
from math import sqrt


In [2]:
# Loading output of VGG Image Annotation tool and create a dataframe

region_data = []
r_data = pd.DataFrame()
r_data_master = pd.DataFrame()

for i in range(1, 27):
    via_folder = '/Users/subrata/workstation/jupyterFiles/yolo_data_file/via_region_data_drg_test_' + str(i) + '.csv'
    region = pd.read_csv(via_folder)
    region_data.append(region)

r_data = pd.concat(region_data).reset_index()
print(len(r_data))
r_data_master = r_data[~r_data['region_shape_attributes'].str.contains('{}')].reset_index(drop=True) # remove drgs having no bbox info
# r_data_master = r_data[~r_data['region_attributes'].str.contains('{}')].reset_index(drop=True) # remove drgs having no bbox info
print(len(r_data_master))  
r_data_master_unique = r_data_master.drop_duplicates('#filename', keep=False).reset_index(drop=True) # remove duplicate rows
print('==========================')
print(len(r_data_master_unique))
r_data_master_unique.drop(r_data_master_unique.columns[[0, 2, 3, 4, 5]], axis=1, inplace=True) # reduce unnecessary columns

num_images_in_data_master = r_data_master_unique["#filename"].nunique()
r_data_master_image_list = r_data_master_unique["#filename"].tolist()

print(num_images_in_data_master)
r_data_master_unique.head()


1428
1428
1406
1406


,#filename,region_shape_attributes,region_attributes
0,1001.jpg,"{""name"":""rect"",""x"":2290,""y"":1257,""width"":1015,...","{""text"":""bom""}"
1,1002.jpg,"{""name"":""rect"",""x"":2171,""y"":1401,""width"":1142,...","{""text"":""bom""}"
2,1003.jpg,"{""name"":""rect"",""x"":2094,""y"":1224,""width"":1215,...","{""text"":""bom""}"
3,1004.jpg,"{""name"":""rect"",""x"":2284,""y"":1648,""width"":1025,...","{""text"":""bom""}"
4,1005.jpg,"{""name"":""rect"",""x"":2280,""y"":1700,""width"":1029,...","{""text"":""bom""}"


In [3]:
## make an image list

num_image_folder = []
drg_image = []

for i in range(1, 27):
    drg_folder = 'drg_test_' + str(i)
    drg_directory = '/Users/subrata/workstation/jupyterFiles/yolo_data_file/' + drg_folder

    drg_folder = [os.path.join(drg_directory, f) for f in os.listdir(drg_directory)if not f.startswith('.')]  # Ignore hidden files like .DS_Store
 #   inter_var_1 = drg_directory + '/{}'
 #   drg_folder = [inter_var_1.format(i) for i in os.listdir(drg_directory)]
    num_image_folder.append(len(drg_folder))
    drg_image.append(drg_folder)

print('Number of images in folders = ', num_image_folder)
num_images = len(drg_image)
print('Number of image folders = ', num_images)


Number of images in folders =  [35, 35, 36, 39, 35, 39, 35, 53, 54, 40, 106, 54, 61, 53, 98, 97, 95, 99, 12, 45, 45, 45, 47, 46, 66, 58]
Number of image folders =  26


In [4]:
## drg_image above is a list of lists. We need to flatten this and create a dataframe: 

from itertools import chain

drg_image_path_df_original = pd.DataFrame()
k = []
drg_image_1 = []

k = list(chain.from_iterable(map(list, drg_image)))
drg_image_1 = k
drg_image_1.sort() # Sorting the list

print(len(drg_image_1))

drg_image_path_values = pd.Series(drg_image_1)
drg_image_path_df_original.insert(loc=0, column='i_path', value = drg_image_path_values)

print(len(drg_image_path_df_original))
drg_image_path_df_original.tail(3)

1428
1428


,i_path
1425,/Users/subrata/workstation/jupyterFiles/yolo_d...
1426,/Users/subrata/workstation/jupyterFiles/yolo_d...
1427,/Users/subrata/workstation/jupyterFiles/yolo_d...


In [5]:
# remove duplicate drawings across folders and create a column for only drg. no.

drg_image_path_df = pd.DataFrame()

drg_image_path_df_original['drg_no'] = drg_image_path_df_original['i_path'].apply(lambda x: x.split('/')[-1])

print(len(drg_image_path_df_original))
print('===========================')
#drg_image_path_df = drg_image_path_df_original.drop_duplicates(subset='drg_no').reset_index(drop=True)
#print(len(drg_image_path_df))
# drg_image_path_df = drg_image_path_df.drop('drg_no', axis=1)
# drg_image_path_df.tail(3)
drg_image_path_df = drg_image_path_df_original.drop_duplicates('drg_no', keep=False).reset_index(drop=True) # remove duplicate rows
print(len(drg_image_path_df))
# Check if '#filename' column in r_data_master_unique and 'drg_no' column in drg_image_path_df are having identical values in all rows:

# Convert columns to sets
set_df1 = set(r_data_master_unique['#filename'])
set_df2 = set(drg_image_path_df['drg_no'])

# Find mismatches
extra_in_df1 = set_df1 - set_df2  # Values present in df1 but not in df2
extra_in_df2 = set_df2 - set_df1  # Values present in df2 but not in df1

# Display results
if not extra_in_df1 and not extra_in_df2:
    print("✅ All values match (ignoring order)!")
else:
    print("❌ Mismatches found:")
    if extra_in_df1:
        print(f"Present in df1 but missing in df2: {extra_in_df1}")
    if extra_in_df2:
        print(f"Present in df2 but missing in df1: {extra_in_df2}")


1428
1406
✅ All values match (ignoring order)!


In [6]:
# merge two data frames on drg.no. or file name and create a combined dataframe with required columns :
merged_drg_data = pd.DataFrame()
merged_drg_data = pd.merge(drg_image_path_df.rename(columns={'drg_no': '#filename'}), r_data_master_unique, on='#filename')
merged_drg_data.tail(3)


,i_path,#filename,region_shape_attributes,region_attributes
1403,/Users/subrata/workstation/jupyterFiles/yolo_d...,10834-2-20-SWZ-01-11-Model-page-001.jpg,"{""name"":""rect"",""x"":1161,""y"":513,""width"":500,""h...","{""text"":""bom""}"
1404,/Users/subrata/workstation/jupyterFiles/yolo_d...,10834-2-20-SWZ-04-00-Model-page-001.jpg,"{""name"":""rect"",""x"":1051,""y"":709,""width"":608,""h...","{""text"":""bom""}"
1405,/Users/subrata/workstation/jupyterFiles/yolo_d...,10834-2-20-SWZ-04-02-Model-page-001.jpg,"{""name"":""rect"",""x"":1138,""y"":874,""width"":517,""h...","{""text"":""bom""}"


In [7]:
# Making a dataframe for Image_id, image path, x, y, width, height, class, image_width, image_height :

x = []  # co-ordinate of bbox left-top corner (and NOT center of bbox) as given by via
y = []  # co-ordinate of bbox left-top corner (and NOT center of bbox) as given by via
width = []  # of boundary box
height = []  # of boundary box
obj_class = []
i_width = []
i_height = []
img_path = []

for i in range(len(merged_drg_data)):
    
    # for x, y, width, height of boundary box:
    r_size = merged_drg_data.values[i, 2][1:(len(merged_drg_data.values[i, 2])-1)]
    r_size_par = r_size.split(",")
    x.append(int("".join(filter(str.isdigit, r_size_par[1]))))
    y.append(int("".join(filter(str.isdigit, r_size_par[2]))))
    width.append(int("".join(filter(str.isdigit, r_size_par[3]))))
    height.append(int("".join(filter(str.isdigit, r_size_par[4]))))
    
    # for object class :
    r_attribs = merged_drg_data.values[i, 3][1:(len(merged_drg_data.values[i, 3])-1)]
    if r_attribs == '':
        this_class = 0
    else:
        this_class = 1
    obj_class.append(this_class)
    
    # for image width, height :
    foto_path = merged_drg_data.values[i,0]
    bgr_image = cv2.imread(foto_path)
    image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
    i_ht, i_wd, _ = image.shape
    i_height.append(i_ht)
    i_width.append(i_wd)

# making the dataframe :    
x_values = pd.Series(x)
y_values = pd.Series(y)
width_values = pd.Series(width)
height_values = pd.Series(height)
class_values = pd.Series(obj_class)
i_width_values = pd.Series(i_width)
i_height_values = pd.Series(i_height)

merged_drg_data.insert(loc=2, column='x', value=x_values)
merged_drg_data.insert(loc=3, column='y', value=y_values)
merged_drg_data.insert(loc=4, column='width', value=width_values)
merged_drg_data.insert(loc=5, column='height', value=height_values)
merged_drg_data.insert(loc=6, column='obj_class', value=class_values)
merged_drg_data.insert(loc=7, column='img_wd', value=i_width_values)
merged_drg_data.insert(loc=8, column='img_ht', value=i_height_values)

merged_drg_data.drop(merged_drg_data.columns[[9, 10]], axis=1, inplace=True) # reduce unnecessary columns
merged_drg_data.rename({'#filename': 'img_id'}, axis=1, inplace=True) # changing column name

merged_drg_data.tail(3)


,i_path,img_id,x,y,width,height,obj_class,img_wd,img_ht
1403,/Users/subrata/workstation/jupyterFiles/yolo_d...,10834-2-20-SWZ-01-11-Model-page-001.jpg,1161,513,500,426,1,1754,1240
1404,/Users/subrata/workstation/jupyterFiles/yolo_d...,10834-2-20-SWZ-04-00-Model-page-001.jpg,1051,709,608,229,1,1754,1240
1405,/Users/subrata/workstation/jupyterFiles/yolo_d...,10834-2-20-SWZ-04-02-Model-page-001.jpg,1138,874,517,71,1,1754,1240


In [8]:
# find drawing of maximum size
df = merged_drg_data.sort_values('img_wd', ascending=False)
df.head(3)

,i_path,img_id,x,y,width,height,obj_class,img_wd,img_ht
332,/Users/subrata/workstation/jupyterFiles/yolo_d...,5305-7-14-FRF-01-03 SH1-Model-page-001.jpg,5894,2774,3511,2302,1,10200,6600
335,/Users/subrata/workstation/jupyterFiles/yolo_d...,5305-7-14-FRF-01-06 SH1-Model-page-001.jpg,7345,2855,2153,2648,1,10200,6600
336,/Users/subrata/workstation/jupyterFiles/yolo_d...,5305-7-14-FRF-01-07-Model-page-001.jpg,7391,4191,2107,1335,1,10200,6600


In [9]:
#since there is huge gap between highest and the next size and qty of highest size is only 3, we shall
# discard these three for the time being 
merged_drg_data.drop(index=[332, 335, 336], inplace=True)
merged_drg_data.reset_index(drop=True, inplace=True)
merged_drg_data.tail(3)

,i_path,img_id,x,y,width,height,obj_class,img_wd,img_ht
1400,/Users/subrata/workstation/jupyterFiles/yolo_d...,10834-2-20-SWZ-01-11-Model-page-001.jpg,1161,513,500,426,1,1754,1240
1401,/Users/subrata/workstation/jupyterFiles/yolo_d...,10834-2-20-SWZ-04-00-Model-page-001.jpg,1051,709,608,229,1,1754,1240
1402,/Users/subrata/workstation/jupyterFiles/yolo_d...,10834-2-20-SWZ-04-02-Model-page-001.jpg,1138,874,517,71,1,1754,1240


In [10]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # Load pretrained model


In [11]:
# Define the function to convert and save YOLO format

def convert_to_yolo_format(df, save_path):
    os.makedirs(save_path, exist_ok=True)
    
    for _, row in df.iterrows():
        img_path = row["i_path"]
        image_name = os.path.basename(img_path)
        label_file = os.path.splitext(image_name)[0] + ".txt"

        # Extract bbox information
        x, y, width, height = row["x"], row["y"], row["width"], row["height"]
        img_wd, img_ht = row["img_wd"], row["img_ht"]
        obj_class = row["obj_class"]
        
        # Convert to YOLO format (normalized)
        x_center = (x + width / 2) / img_wd
        y_center = (y + height / 2) / img_ht
        norm_width = width / img_wd
        norm_height = height / img_ht
        
        # Ensure values are in [0, 1] range
        yolo_data = f"{obj_class} {x_center:.6f} {y_center:.6f} {norm_width:.6f} {norm_height:.6f}\n"
        
        # Save to label file
        with open(os.path.join(save_path, label_file), "a") as f:
            f.write(yolo_data)
        
# Convert and save annotations
convert_to_yolo_format(merged_drg_data, "/Users/subrata/workstation/jupyterFiles/yolo_data_file/all_labels")

print("YOLO annotations saved in 'all_labels' directory.")

YOLO annotations saved in 'all_labels' directory.


In [12]:
# Make all drawings directory named "all_images"

for i in range(len(merged_drg_data)):

    image_path = merged_drg_data.values[i,0]
    image_id = merged_drg_data.values[i,1].split(".")[0]
    x = cv2.imread(image_path)

    # Define output filename dynamically
    output_filename = f"/Users/subrata/workstation/jupyterFiles/yolo_data_file/all_images/{image_id}.jpg"

    # Save image
    cv2.imwrite(output_filename, x)


In [13]:
# Splitting data in train and validation sets :

from sklearn.model_selection import train_test_split
import shutil

# Define paths
image_dir = "/Users/subrata/workstation/jupyterFiles/yolo_data_file/all_images"
label_dir = "/Users/subrata/workstation/jupyterFiles/yolo_data_file/all_labels"
train_img_dir = "/Users/subrata/workstation/jupyterFiles/yolo_data_file/yolov8_dir/images/train"
val_img_dir = "/Users/subrata/workstation/jupyterFiles/yolo_data_file/yolov8_dir/images/val"
train_label_dir = "/Users/subrata/workstation/jupyterFiles/yolo_data_file/yolov8_dir/labels/train"
val_label_dir = "/Users/subrata/workstation/jupyterFiles/yolo_data_file/yolov8_dir/labels/val"

# Create directories
for path in [train_img_dir, val_img_dir, train_label_dir, val_label_dir]:
    os.makedirs(path, exist_ok=True)

# Load image paths
image_paths = merged_drg_data["i_path"].tolist()

# Split data
train_images, val_images = train_test_split(image_paths, test_size=0.2, random_state=42)

# Move images and labels
def move_files(image_list, img_dest, label_dest):
    for img_path in image_list:
        img_name = os.path.basename(img_path)
        label_name = os.path.splitext(img_name)[0] + ".txt"
        
        # Move image
        shutil.copy(img_path, os.path.join(img_dest, img_name))
        
        # Move label file
        if os.path.exists(os.path.join(label_dir, label_name)):
            shutil.copy(os.path.join(label_dir, label_name), os.path.join(label_dest, label_name))

move_files(train_images, train_img_dir, train_label_dir)
move_files(val_images, val_img_dir, val_label_dir)

print("Data successfully split into train and validation sets.")

Data successfully split into train and validation sets.


In [14]:
# YOLOv8 requires a YAML configuration file specifying dataset paths
# Creating data.yaml file for feeding the training module

yaml_content = """train: /Users/subrata/workstation/jupyterFiles/yolo_data_file/yolov8_dir/images/train
val: /Users/subrata/workstation/jupyterFiles/yolo_data_file/yolov8_dir/images/val

nc: 2
names: ["n_bom", "bom"]
"""

with open("/Users/subrata/workstation/jupyterFiles/yolo_data_file/yolov8_dir/data.yaml", "w") as file:
    file.write(yaml_content)

print("data.yaml created successfully!")


data.yaml created successfully!


In [ ]:
# Train the model
model.train(
    data="/Users/subrata/workstation/jupyterFiles/yolo_data_file/yolov8_dir/data.yaml", 
    epochs=10, 
    imgsz=640, 
    batch=8, 
    workers=2, 
    device="cpu"  # Since CUDA is not available
)

print("Training completed!")

In [ ]:
## Execution of above cell will do the training.

In [8]:
import shutil
# Define source (where YOLO saves the trained model)
source_path = "runs/detect/train9/weights/best.pt"

# Define destination (where you want to store it)
destination_path = "/Users/subrata/workstation/jupyterFiles/yolo_data_file/yolov8_dir/my_yolo_model_1.pt"

# Copy the model
shutil.copy(source_path, destination_path)

print(f"Model saved to: {destination_path}")

Model saved to: /Users/subrata/workstation/jupyterFiles/yolo_data_file/yolov8_dir/my_yolo_model_1.pt
